This notebook performs filtering on the PCAWG cohort to remove samples with technical artifacts that greatly impair HOR size estimation

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import Counter
from kneed import KneeLocator

In [ ]:
chromosomes = ["chr1", "chr3", "chr4", "chr5", "chr6", "chr7", "chr8", "chr9", "chr10", "chr11", "chr12", "chr16", "chr17", "chr18", "chr19", "chr20"] 

## Load PCAWG prediciton data
- the data is already reformatted to include germline and tumor predictions in the same row
- the length prediction was done based on a CN=2 level
- the data already contains CN classification

In [ ]:
df = pd.read_csv("/g/korbel/hain/centromere/cancer/horoscope_results.pcawg.no_metadata.tsv", sep="\t")

In [ ]:
df["LOG_LENGTH_2"] = np.log(df["TUMOR_TOTAL_HOR_LENGTH_COV_2"] / df["GERMLINE_TOTAL_HOR_LENGTH_COV_2"])

#### Remove WGD samples
- WGD samples are hard to analyse as their copy number changes, their general state is not clearly defined and their CNA profile is generally highly unstable

In [ ]:
df_nowgd = df[
    (~pd.isna(df["LOG_LENGTH_2"])) &    # remove samples without full HOR size predictions -> two samples without DBG data
    (df["WGD"] == "no_wgd")             # remove WGD
].copy()

In [ ]:
df_nowgd["SAMPLE_TUMOR"].nunique()

#### Check how many copy neutral chromosomes have similiar HOR length between the tumor and germline sample

In [ ]:
cutoff_analysis_data = []
n_cn_neutral_df = pd.DataFrame()
frac_exact_df = pd.DataFrame()

for cutoff in tqdm(np.linspace(0.0, 0.5, 11)):
    
    passing_samples = 0
    storage_frac_exact = []
    storage_n_cn_neutral = []
    
    for sample, df_sample in df_nowgd.groupby("SAMPLE_TUMOR"): # type: ignore

        # count number of cn neutral chromosomes
        num_neutral = df_sample[df_sample["CN"] == "2_2"].shape[0]
        # count number of cn neutral chromosomes with similiar HOR lengths between germline and tumor
        num_neutral_similar = df_sample[
            (df_sample["CN"] == "2_2") & 
            df_sample["LOG_LENGTH_2"].between(-cutoff, cutoff)
        ].shape[0]
        
        # only consider samples with at least 4 cn neutral chromosomes
        if num_neutral > 3:
            frac_exact = num_neutral_similar / num_neutral
            N = num_neutral
        else:
            frac_exact = 0
            N = 0

        if frac_exact > 0.5:
            passing_samples += 1
        storage_frac_exact.append(frac_exact)
        storage_n_cn_neutral.append(N)

    cutoff_analysis_data.append({
        "cutoff": cutoff,
        "n_samples": passing_samples
    })
    frac_exact_df[cutoff] = storage_frac_exact
    n_cn_neutral_df[cutoff] = storage_n_cn_neutral

    
cutoff_analysis_df = pd.DataFrame(cutoff_analysis_data)

In [ ]:
ax1 = plt.subplot(1, 2, 1)
for col in frac_exact_df.columns:
    ax1.hist(frac_exact_df[col], histtype="step", density=True, cumulative=True, bins=np.linspace(0, 1, 100), label=str(np.round(col, 2))) # type: ignore
ax1.legend(title="Cutoff")
ax1.set_title("CDist of Fraction Exact by Cutoff")
ax1.set_xlabel("Fraction Exact")
ax1.set_ylabel("Cumulative Density")

ax2 = plt.subplot(1, 2, 2)
sns.histplot(n_cn_neutral_df[0], discrete=True, cumulative=True, ax=ax2) # type: ignore
ax2.set_title("CDist of Number of CN Neutral Chromosomes")
ax2.set_xlabel("Number of CN Neutral Chromosomes")
ax2.set_ylabel("Cumulative Density")

plt.gcf().set_size_inches(12, 4)
plt.show()

#### Make columns indicating different cutoffs
4+ and more than 50% of CN-neutral chromosomes show HOR length ration between tumor and germline sample bsmaller than the threshold

In [ ]:
# define multiple cutoffs
cutoff_groups = {
    0.12:"GROUP1", 0.1:"GROUP2", 0.2:"GROUP3", 0.3:"GROUP4", 0.4:"GROUP5", 
    0.5:"GROUP6", 0.15:"GROUP7", 0.25:"GROUP8", 0.21:"GROUP9", 0.22:"GROUP10",
    0.23:"GROUP11", 0.24:"GROUP12"
    }

# subset to neutral chromosomes
neutral = df_nowgd[df_nowgd["CN"] == "2_2"]

flags = {}
for cutoff, group in cutoff_groups.items():
    frac = (
        neutral.assign(ok=neutral["LOG_LENGTH_2"].abs() < cutoff)
        .groupby("SAMPLE_TUMOR")["ok"].mean()
    )
    n = (
        neutral.groupby("SAMPLE_TUMOR")["CHROM"].count()
    )
    
    flags[group] = (frac > 0.5) & (n > 3)

result = pd.DataFrame(flags)

In [ ]:
### add cutoff columsn to DF
for cutoff, group in cutoff_groups.items():
    col_cutoff = []
    for _, row in df_nowgd.iterrows(): # type: ignore
        if result.loc[row["SAMPLE_TUMOR"], group]: # type: ignore
            col_cutoff.append(True)
        else:
            col_cutoff.append(False)
    df_nowgd[group] = col_cutoff

### Diagnostics: Check fraction of passing/failing samples per Histology and DCC project in PCAWG

In [ ]:
cross_dcc_filter = pd.crosstab(
    df_nowgd["DCC_PROJECT_CODE"],
    df_nowgd["GROUP4"]
)

In [ ]:
cross_dcc_filter = pd.crosstab(
    df_nowgd["DCC_PROJECT_CODE"],
    df_nowgd["GROUP4"]
)
cross_dcc_filter["FRAC_FILTERED"] = cross_dcc_filter[False] / (cross_dcc_filter[True] + cross_dcc_filter[False])

In [ ]:
cross_hist_filter = pd.crosstab(
    df_nowgd["HISTOLOGY"],
    df_nowgd["GROUP4"]
)
cross_hist_filter["FRAC_FILTERED"] = cross_hist_filter[False] / (cross_hist_filter[True] + cross_hist_filter[False])

In [ ]:
fig, ax = plt.subplots(figsize=(20, 4))
sns.barplot(
    data=cross_dcc_filter.sort_values(by="FRAC_FILTERED"),
    x=cross_dcc_filter.sort_values(by="FRAC_FILTERED").index,
    y="FRAC_FILTERED",
    ax=ax
)
ax.set_ylim(0, 1)
plt.xticks(rotation=90)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))

x = 0
xticks = []
xticklabels = []
for histology in cross_hist_filter.sort_values(by="FRAC_FILTERED").index:
    dccs = df_nowgd[df_nowgd["HISTOLOGY"] == histology]["DCC_PROJECT_CODE"].unique()
    
    bar_data = []
    for dcc in dccs:
        bar_data.append(
            df_nowgd[
                (df_nowgd['DCC_PROJECT_CODE'] == dcc) &
                (df_nowgd['HISTOLOGY'] == histology) &
                (df_nowgd['GROUP4'] == False)
            ]['SAMPLE_TUMOR'].nunique() / 
            df_nowgd[
                (df_nowgd['DCC_PROJECT_CODE'] == dcc) &
                (df_nowgd['HISTOLOGY'] == histology)
            ]['SAMPLE_TUMOR'].nunique()
        )
        
    ax.bar(x=range(x, x + len(dccs)), height=bar_data)

    for i, dcc in enumerate(dccs):
        n_samples = df_nowgd[
            (df_nowgd['DCC_PROJECT_CODE'] == dcc) &
            (df_nowgd['HISTOLOGY'] == histology)
            ]['SAMPLE_TUMOR'].nunique()
        
        ax.text(x + i, bar_data[i] + 0.01, f"{dcc} n={n_samples}", rotation=90, va='bottom', ha="center")

    
    xticks.append(x + (len(dccs) - 1) / 2)
    xticklabels.append(histology)
    x += len(dccs)

ax.set_xticks(xticks)
ax.set_xticklabels(xticklabels, rotation=90)
ax.set_title("Fraction of filtered sample per Histology / DCC project code")
ax.set_xlabel("Histolgy")
ax.set_ylabel("Fraction filtered")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("../plotting/plots/pcawg_filtering.frac_filtered_by_histology_and_dcc.png")
plt.show()

In [ ]:
k = 1
dcc_list = cross_dcc_filter.sort_values(by="FRAC_FILTERED").index.tolist()[::-1]
dcc_list = cross_dcc_filter.index.tolist()[::-1]

min_y = 10
max_y = -10

for dcc in dcc_list:
    dcc_df = df_nowgd[(df_nowgd["CN"] == "2_2") & (df_nowgd["DCC_PROJECT_CODE"] == dcc)]
    ax = plt.subplot(10, 5, k)
    ax.set_title(f"DCC: {dcc}")
    for filter in [True, False]:
        
        sample_order = dcc_df[dcc_df["GROUP4"] == filter].groupby("SAMPLE_TUMOR")["LOG_LENGTH_2"].mean().sort_values().keys().tolist()
        sns.boxplot(
            data=dcc_df[dcc_df["GROUP4"] == filter],
            x="SAMPLE_TUMOR",
            y="LOG_LENGTH_2",
            order=sample_order,
            showfliers=False,
            boxprops={"edgecolor": "none"}
        )
    ax.set_xticks([])
    ax.set_xlabel("Samples")
    ax.set_ylabel("log(T/N)")
    min_y = min(min_y, ax.get_ylim()[0])
    max_y = max(max_y, ax.get_ylim()[1])
    ax.set_ylim(2, -2)
    ax.set_yticks([-2, 0, 2])
    ax.plot(
        [-0.5, dcc_df["SAMPLE_TUMOR"].nunique()-0.5], 
        [0, 0], 
        color="gray", linestyle="--")
    ax.plot(
        [dcc_df[dcc_df["GROUP4"] == True]["SAMPLE_TUMOR"].nunique()-0.5, dcc_df[dcc_df["GROUP4"] == True]["SAMPLE_TUMOR"].nunique()-0.5], 
        [-1, 1], 
        color="gray", linestyle="--")
    k += 1
    if k == 6:
        pass#break

plt.gcf().set_size_inches(20, 40)
plt.tight_layout()
plt.savefig("../plotting/plots/pcawg_filtering.logTN_by_DCC.png")

## Calculate F - the number of centromeres per tumor cell
- $F=1 + 2/p*((T/N)-1)$
- p = tumor purity
- T = tumor sample HOR size
- N = germline sample HOR size

In [ ]:
df_nowgd["TOTAL_HOR_LENGTH_F_COV_2"] = 1 + 2 / df_nowgd["PURITY"] * ((df_nowgd["TUMOR_TOTAL_HOR_LENGTH_COV_2"] / df_nowgd["GERMLINE_TOTAL_HOR_LENGTH_COV_2"]) - 1)

In [ ]:
output_columns = [
    'SAMPLE_TUMOR',
    'SAMPLE_GERMLINE',
    'CHROM',
    'HISTOLOGY',
    'DCC_PROJECT_CODE',
    'CN',
    'PURITY',
    'CLUSTER_H1',
    'CLUSTER_H2',
    'TOTAL_HOR_LENGTH_F_COV_2'
]  + [x for x in df_nowgd.columns if "GROUP" in x] # type: ignore

In [ ]:
df_nowgd[output_columns].to_csv("pcawg_prediction.tsv", sep="\t", index=False)